# DE-LU Day-Ahead Futures Hackathon

This notebook walks through the official naive-copy baseline for the student challenge.

## 1. Challenge Objective

Build a strategy that chooses `+1`, `-1`, or `None` each day using only daily information available at decision time.

## 2. Set Up the Notebook Environment

In [1]:
from pathlib import Path
import sys

def find_repo_root() -> Path:
    candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
    for candidate in candidates:
        if (candidate / "pyproject.toml").exists() and (candidate / "src").exists():
            return candidate.resolve()
    raise FileNotFoundError("Could not locate the repository root from this notebook.")

REPO_ROOT = find_repo_root()
SRC_PATH = REPO_ROOT / "src"

for path in (REPO_ROOT, SRC_PATH):
    path_str = str(path)
    if path_str not in sys.path:
        sys.path.insert(0, path_str)

REPO_ROOT

WindowsPath('C:/Repositories/Energy_Modelling')

## 3. Load the Daily Dataset

In [2]:
import pandas as pd

from energy_modelling.challenge.data import write_challenge_data

data_path = REPO_ROOT / "data" / "challenge" / "daily_public.csv"
if not data_path.exists():
    source_dataset = REPO_ROOT / "kaggle_upload" / "dataset_de_lu.csv"
    if not source_dataset.exists():
        raise FileNotFoundError(
            f"Could not find {data_path} or the source dataset at {source_dataset}."
        )
    write_challenge_data(source_dataset, REPO_ROOT / "data" / "challenge")

daily = pd.read_csv(data_path, parse_dates=["delivery_date"])
daily["delivery_date"] = daily["delivery_date"].dt.date
daily.head()

,delivery_date,split,last_settlement_price,settlement_price,price_change_eur_mwh,target_direction,pnl_long_eur,pnl_short_eur,gen_solar_mw_mean,gen_wind_onshore_mw_mean,...,price_cz_eur_mwh_mean,price_dk_1_eur_mwh_mean,flow_fr_net_import_mw_mean,flow_nl_net_import_mw_mean,carbon_price_usd_mean,gas_price_usd_mean,price_mean,price_max,price_min,price_std
0,2019-01-01,train,28.320000,-6.875833,-35.195833,-1,-844.70,844.70,0.122500,20323.970000,...,8.500000,28.320000,-1044.080000,-1692.000000,NaN,NaN,28.320000,28.32,28.32,NaN
1,2019-01-02,train,-6.875833,29.104167,35.980000,1,863.52,-863.52,405.788333,32035.105000,...,11.240417,-6.504167,-806.186667,-1765.431250,NaN,NaN,-6.875833,10.07,-33.57,10.786639
2,2019-01-03,train,29.104167,58.042083,28.937917,1,694.51,-694.51,1252.363646,24710.584375,...,39.795417,30.414167,-183.780000,-1243.382083,17.924999,22.475000,29.104167,62.11,-48.93,40.083914
3,2019-01-04,train,58.042083,48.917083,-9.125000,-1,-219.00,219.00,920.588021,9366.417187,...,59.270417,56.897500,-1179.576667,-1959.829583,16.642500,22.300833,58.042083,69.55,43.88,9.148248
4,2019-01-05,train,48.917083,43.155417,-5.761667,-1,-138.28,138.28,392.281250,19177.198750,...,52.396250,48.615417,-1208.992917,-1666.769583,16.875000,22.789375,48.917083,55.78,26.90,7.091700


## 4. Understand the Split and Target

In [3]:
daily[["split", "last_settlement_price", "settlement_price", "target_direction"]].head()

,split,last_settlement_price,settlement_price,target_direction
0,train,28.320000,-6.875833,-1
1,train,-6.875833,29.104167,1
2,train,29.104167,58.042083,1
3,train,58.042083,48.917083,-1
4,train,48.917083,43.155417,-1


In [4]:
daily["split"].value_counts().sort_index()

split
train         1826
validation     366
Name: count, dtype: int64

## 5. Create the Public Train and Validation Sets

In [5]:
train = daily[daily["split"] == "train"].copy()
validation = daily[daily["split"] == "validation"].copy()
len(train), len(validation)

(1826, 366)

## 6. Build the Naive Copy Baseline

The naive baseline always goes long.

In [6]:
class NaiveCopyStrategy:
    def fit(self, train_data: pd.DataFrame) -> None:
        self.training_rows = len(train_data)

    def act(self, state) -> int | None:
        return 1

    def reset(self) -> None:
        pass

## 7. Backtest the Baseline on 2024

In [7]:
from datetime import date

from energy_modelling.challenge.runner import run_challenge_backtest

strategy = NaiveCopyStrategy()
result = run_challenge_backtest(
    strategy=strategy,
    daily_data=daily,
    training_end=date(2023, 12, 31),
    evaluation_start=date(2024, 1, 1),
    evaluation_end=date(2024, 12, 31),
)
result.metrics

{'total_pnl': 1240.9000000000028,
 'days_evaluated': 366.0,
 'trade_count': 366.0,
 'sharpe_ratio': 0.07163789508582825,
 'max_drawdown': 8992.45,
 'win_rate': 0.4644808743169399,
 'avg_win': 572.5159411764706,
 'avg_loss': -490.2388265306123,
 'best_day': 3092.6000000000004,
 'worst_day': -5225.9}

## 8. Inspect Daily Predictions and PnL

In [8]:
pd.DataFrame({
    "prediction": result.predictions,
    "daily_pnl": result.daily_pnl,
    "cumulative_pnl": result.cumulative_pnl,
}).head()

,prediction,daily_pnl,cumulative_pnl
2024-01-01,1,217.62,217.62
2024-01-02,1,824.24,1041.86
2024-01-03,1,-100.55,941.31
2024-01-04,1,936.08,1877.39
2024-01-05,1,110.44,1987.83


## 9. Turn the Baseline into a Submission Class

Copy the same logic into `submission/student_strategy.py`, then replace the naive decision rule with your own model or heuristic.

In [9]:
from submission.student_strategy import StudentStrategy

submission_strategy = StudentStrategy()
submission_strategy.fit(train)
submission_strategy.training_rows

1826

## 10. Ideas for Improvement

Try momentum, mean reversion, weather-driven rules, or a simple classifier trained on the lagged daily features.